# Tenent Settings Best Practics Analyzer

When you run this notebook, the Tenant Settings Best Practice Analyzer (TSBPA) will offer tips to improve your tenant settings.

The TSBPA checks against recommendations for the current (2026-06-01) 167 tenant settings. These recommendations come from experts within the Fabric Community.

You’ll get suggestions for improvement in the following categories: 
* Additional workloads
* Admin API settings
* Advanced networking
* App settings
* Audit and usage settings
* Azure AI Service
* Azure Maps services
* Copilot and Azure OpenAI Service
* Dashboard settings
* Datamart settings
* Developer settings
* Discovery settings
* Domain management settings
* Encryption
* Explore settings (preview)
* Export and sharing settings
* Gen1 dataflow settings
* Git integration
* Help and support settings
* Information protection
* Insights settings
* Integration settings
* Microsoft Fabric
* OneLake settings
* Power BI visuals
* Q&A settings
* R and Python visuals settings
* Scale-out settings
* Scorecards settings
* Semantic Model Security
* Semantic model settings
* Share data with your Microsoft 365 services
* Template app settings
* User experience experiments
* Workspace settings

## Powering this feature: Semantic Link (Lab)
This notebook leverages [Semantic Link](https://learn.microsoft.com/fabric/data-science/semantic-link-overview) and [Semantic Link Lab Admin](https://semantic-link-labs.readthedocs.io/en/stable/sempy_labs.admin.html), python libraries which lets you query and update  Fabric items for different use cases. The "[list_tenant_settings](https://semantic-link-labs.readthedocs.io/en/stable/sempy_labs.admin.html#sempy_labs.admin.list_tenant_settings)" and "[update_tenant_setting](https://semantic-link-labs.readthedocs.io/en/stable/sempy_labs.admin.html#sempy_labs.admin.update_tenant_setting)" functions used in this notebook are just one example of the useful [functions]((https://learn.microsoft.com/python/api/semantic-link-sempy/sempy.fabric)) which Semantic Link and Semantic Link Labs offers.

You can find more [functions](https://github.com/microsoft/semantic-link-labs#featured-scenarios) and [helper notebooks](https://github.com/microsoft/semantic-link-labs/tree/main/notebooks) in [Semantic Link Labs](https://github.com/microsoft/semantic-link-labs), a Python library that extends Semantic Link's capabilities to automate technical tasks.

## Low-code solutions for data tasks
You don't have to be a Python expert to use Semantic Link or Semantic Link Labs. Many functions can be used simply by entering your parameters and running the notebook.


## Install and import libraries

In [9]:
# Install SemPy
%pip install semantic-link
%pip install semantic-link-labs


StatementMeta(, 6335fa0a-66d3-4c02-97d3-63d46223ad03, 24, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [10]:
# import the module
from sempy_labs import admin 
import pandas as pd

StatementMeta(, 6335fa0a-66d3-4c02-97d3-63d46223ad03, 26, Finished, Available, Finished, False)

## Parameter

In [11]:
recommendation_mode = "Light" # "Light" or "Paranoid"
recommendation_path_and_file = "/lakehouse/default/Files/tenant_settings_recommendations.csv"
recommendation_file_delimiter = ";"
debug_mode = False

StatementMeta(, 6335fa0a-66d3-4c02-97d3-63d46223ad03, 27, Finished, Available, Finished, False)

## Defaults for missing parameters

In [12]:
#recommendation_mode = globals().get("recommendation_mode", "Light")
#recommendation_path_and_file = globals().get("recommendation_path_and_file", "/lakehouse/default/Files/tenant_settings_recommendations.csv")
#recommendation_file_delimiter = globals().get("recommendation_file_delimiter", ";")
#debug_mode = globals().get("debug_mode", "False")
 
print(f"recommendation_mode: '{recommendation_mode}'")
print(f"recommendation_path_and_file: '{recommendation_path_and_file}'")
print(f"recommendation_file_delimiter: '{recommendation_file_delimiter}'")
print(f"debug_mode: '{debug_mode}'")

# Derive variables from parameter
recommendation = f"Best Practice Recommendation {recommendation_mode}"
print(f"recommendation: '{recommendation}'")

StatementMeta(, 6335fa0a-66d3-4c02-97d3-63d46223ad03, 28, Finished, Available, Finished, False)

recommendation_mode: 'Light'
recommendation_path_and_file: '/lakehouse/default/Files/tenant_settings_recommendations.csv'
recommendation_file_delimiter: ';'
debug_mode: 'False'
recommendation: 'Best Practice Recommendation Light'


## Ingest general recommendations from CSV

In [22]:
tenant_settings_recommendation = pd.read_csv(
    recommendation_path_and_file, 
    delimiter = recommendation_file_delimiter)
if debug_mode==True:
    display(tenant_settings_recommendation)


StatementMeta(, 6335fa0a-66d3-4c02-97d3-63d46223ad03, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a496f316-2aad-43b0-b689-453ef7875ebc)

## Define general function 

In [17]:
def get_recommended_settings_script(
    level,
    tenant_settings,
    tenant_settings_recommendation,
    recommendation,
    check_missing_settings,
    function_name,
    setting_name,
    title_settings_name,
    enabled_settings_name,
    debug_mode=False
):
    """
    Processes tenant settings, compares with recommendation data,
    and prints update commands for mismatched settings.
    """
    if tenant_settings.empty:
        print(f"No settings for {level}, no recommendations.")
        return
    else:
        print(f"# Recommended changes for your {level} settings:")


    # Merge settings with recommendation
    tenant_settings = tenant_settings.merge(
        tenant_settings_recommendation,
        on=setting_name,
        how="left",
        suffixes=("_settings", "_recommendation")
    )

    if debug_mode:
        display(tenant_settings)

    # Find missing recommendations
    tenant_settings_missing = tenant_settings.loc[
        tenant_settings["Title_recommendation"].isna()
        | tenant_settings["Title_recommendation"].eq(""),
        #["Setting Name", "Title_settings", "Enabled_settings"]
        [setting_name, title_settings_name, enabled_settings_name]
    ]

    if not tenant_settings_missing.empty and check_missing_settings == True:
        print("##################################################################")
        print("# The recommendation file does not contain the following settings:")
        print(f"# {tenant_settings_missing}")
        print("# Take a close look at these settings manually.")
        print("##################################################################")
    #else:
        #print("You're all set!")


    # Print install instructions
    print("# Install SemPy")
    print("%pip install semantic-link-labs\n")
    print("# import the module")
    print("from sempy_labs import admin\n")

    # Filter settings
    col = tenant_settings[f"{recommendation}"]
    tenant_settings_filtered = tenant_settings.loc[
        col.notna()
        & col.astype(str).ne("")
        & col.astype(str).str.lower().ne("n/a")
        & tenant_settings[enabled_settings_name].ne(col),
        #["Setting Name", "Title_settings", "Enabled_settings", recommendation]
        [setting_name, title_settings_name, enabled_settings_name, recommendation]
    ]

    # Generate update commands
    for _, tenant_setting in tenant_settings_filtered.iterrows():
        if level != "" and level != "Tenant":
            id_column_name = f'{level} Id'
            id_column_text = f'"{tenant_setting[id_column_name]}", '
        else:
            id_column_name = ""
            id_column_text = ""
        print(
            f'admin.{function_name}('
            f'{id_column_text}'
            f'"{tenant_setting[setting_name]}", '
            f'"{tenant_setting[recommendation]}") '
            f'# {tenant_setting[title_settings_name]}'
        )

    #return tenant_settings

StatementMeta(, 6335fa0a-66d3-4c02-97d3-63d46223ad03, 33, Finished, Available, Finished, False)

## Generate recommendations

In [15]:
debug_mode = True

StatementMeta(, 6335fa0a-66d3-4c02-97d3-63d46223ad03, 31, Finished, Available, Finished, False)

### Tenant Level

In [23]:
tenant_settings = admin.list_tenant_settings()
if debug_mode==True:
    display(tenant_settings)
get_recommended_settings_script(
    level="Tenant",
    tenant_settings=tenant_settings, 
    tenant_settings_recommendation=tenant_settings_recommendation, 
    recommendation=recommendation, 
    check_missing_settings=True,
    function_name="update_tenant_setting",
    setting_name="Setting Name",
    title_settings_name="Title_settings",
    enabled_settings_name="Enabled_settings",
    debug_mode=debug_mode
)

StatementMeta(, 6335fa0a-66d3-4c02-97d3-63d46223ad03, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cd2a169d-f46b-4d98-9268-f919af03ae34)

# Recommended changes for your Tenant settings:


SynapseWidget(Synapse.DataFrame, 04ee0791-f0d4-45bb-b405-ad09b7c8b65b)

# Install SemPy
%pip install semantic-link-labs

# import the module
from sempy_labs import admin

admin.update_tenant_setting("CertifiedCustomVisualsTenant", "True") # Add and use certified visuals only (block uncertified)
admin.update_tenant_setting("DatamartTenant", "False") # Create Datamarts (preview)
admin.update_tenant_setting("PublishToWeb", "False") # Publish to web


### Capacity level

In [24]:
tenant_settings = admin.list_capacity_tenant_settings_overrides()
if debug_mode==True:
    display(tenant_settings)
get_recommended_settings_script(
    level="Capacity",
    tenant_settings=tenant_settings, 
    tenant_settings_recommendation=tenant_settings_recommendation, 
    recommendation=recommendation, 
    check_missing_settings=True,
    function_name="update_capacity_tenant_setting_override",
    setting_name="Setting Name",
    title_settings_name="Setting Title_settings",
    enabled_settings_name="Setting Enabled_settings",
    debug_mode=True
)

StatementMeta(, 6335fa0a-66d3-4c02-97d3-63d46223ad03, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c5a86029-5690-4707-a0ee-32ba360e7e98)

No settings for Capacity, no recommendations.


### Domain level

In [25]:
tenant_settings = admin.list_domain_tenant_settings_overrides()
if debug_mode==True:
    display(tenant_settings)
get_recommended_settings_script(
    level="Domain",
    tenant_settings=tenant_settings, 
    tenant_settings_recommendation=tenant_settings_recommendation, 
    recommendation=recommendation, 
    check_missing_settings=True,
    function_name="update_domain_tenant_setting_override",
    setting_name="Setting Name",
    title_settings_name="Title_settings",
    enabled_settings_name="Enabled_settings",
    debug_mode=debug_mode
)

StatementMeta(, 6335fa0a-66d3-4c02-97d3-63d46223ad03, 41, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, 046bf0d8-81fd-44a7-9ba7-0054bf7a70aa)

No settings for Domain, no recommendations.


### Workspace level

In [26]:
tenant_settings = admin.list_workspaces_tenant_settings_overrides()
if debug_mode==True:
    display(tenant_settings)
get_recommended_settings_script(
    level="Workspace",
    tenant_settings=tenant_settings, 
    tenant_settings_recommendation=tenant_settings_recommendation, 
    recommendation=recommendation, 
    check_missing_settings=True,
    function_name="update_workspace_tenant_setting_override",
    setting_name="Setting Name",
    title_settings_name="Title_settings",
    enabled_settings_name="Enabled_settings",
    debug_mode=debug_mode
)


StatementMeta(, 6335fa0a-66d3-4c02-97d3-63d46223ad03, 42, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, dd419ba4-637d-4eae-a492-3b7306a63a45)

No settings for Workspace, no recommendations.
